# V7 — FSC-147 + 111 Verified Absent Pairs: Unified Evaluation

## Mục tiêu

Ghép **FSC-147 positive** với **111 hard-negative query–image pairs** và đánh giá V7 một cách không nhầm lẫn giữa số ảnh và số truy vấn.

- FSC-147: 6.135 cặp positive nếu dữ liệu chính thức đầy đủ; GT là số point annotation của target đúng.
- 111 hard-negative: GT count = **0**, trong 96 ảnh đã thuộc FSC-147.
- Dự kiến: **6.246 cặp ảnh–truy vấn**, KHÔNG phải 6.246 ảnh độc lập.
- Benchmark quan trọng nhất: official FSC-147 TEST positive + 111 TEST negative.
- Full FSC-147 + 111 là kiểm tra độ phủ toàn bộ dataset; không gọi train/val/test trộn lẫn là test độc lập.

### Phân biệt BUILD / EVALUATE

Notebook này **tạo manifest + ghép và đánh giá kết quả V7**. Nó KHÔNG tự động chạy Qwen/CountGD++ trên cặp chưa có dự đoán. Muốn có kết quả full phải chạy pipeline V7 cho các cặp trong `v7_pending_predictions.csv` (có thể chia shard); khi đã có các dự đoán, Add Input CSV rồi chạy lại notebook.

**Không điền số 0 cho những cặp thiếu dự đoán.**


In [ ]:
# CELL 1 — CONFIG
from pathlib import Path
import json, csv, zipfile, hashlib
import pandas as pd
import numpy as np
from IPython.display import display

SEED = 42
OUTPUT_DIR = Path('/kaggle/working/nckh_v7_fsc147_111_joint')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chọn TEST cho bảng đánh giá chính; FULL dùng cho bảng thống kê toàn dataset.
# Notebook vẫn tạo cả hai manifest.
NUM_SHARDS = 1
SHARD_INDEX = 0

# Không công bố kết quả chung khi còn thiếu ít nhất một cặp.
REQUIRE_COMPLETE_FOR_MAIN_RESULTS = True

# File dự đoán FULL từ pipeline V7 (có thể chứa nhiều shard).
# Mỗi CSV phải có group_id + v7_count (hoặc pred_count).
PREDICTION_NAME_GLOB = 'v7_*predictions*.csv'

SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# CELL 2 — TÌM ĐÚNG FSC-147 + BENCHMARK 111
# Nếu trước đây Kaggle thiếu 2 file metadata, có thể tải metadata chuẩn từ HF.
from pathlib import Path

def find_file(name, allow_output=False):
    hits = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob(name):
            if p.is_file() and (allow_output or OUTPUT_DIR not in p.parents):
                hits.append(p)
    return sorted(hits, key=lambda p:(0 if str(p).startswith('/kaggle/input') else 1,len(str(p))))[0] if hits else None

def find_image_dir():
    hits=[]
    for root in SEARCH_ROOTS:
        if root.exists():
            hits += [p for p in root.rglob('images_384_VarV2') if p.is_dir()]
    return max(hits, key=lambda p: len(list(p.glob('*.jpg')))) if hits else None

IMAGE_DIR = find_image_dir()
ANNOTATION_FILE = find_file('annotation_FSC147_384.json')
CLASS_FILE = find_file('ImageClasses_FSC147.txt')
SPLIT_FILE = find_file('Train_Test_Val_FSC_147.json')
ABSENT_CSV = find_file('FSC147_Presence_verified_absent_111.csv')

# Metadata-only fallback: dùng đúng các file nhãn / split chính thức.
if CLASS_FILE is None or SPLIT_FILE is None:
    print('Thiếu class/split FSC-147; đang thử tải metadata chuẩn từ Hugging Face...')
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
        from huggingface_hub import hf_hub_download
    if CLASS_FILE is None:
        CLASS_FILE=Path(hf_hub_download('isentropic/FSC147', filename='ImageClasses_FSC147.txt',repo_type='dataset'))
    if SPLIT_FILE is None:
        SPLIT_FILE=Path(hf_hub_download('isentropic/FSC147', filename='Train_Test_Val_FSC_147.json',repo_type='dataset'))

for label,p in [('IMAGE_DIR',IMAGE_DIR),('ANNOTATION_FILE',ANNOTATION_FILE),('CLASS_FILE',CLASS_FILE),('SPLIT_FILE',SPLIT_FILE),('ABSENT_CSV',ABSENT_CSV)]:
    print(label,':',p)
    if p is None or not p.exists():
        raise FileNotFoundError(f'Thiếu {label}. Add FSC147 images/annotation + 111 CSV vào Kaggle Input.')


In [ ]:
# CELL 3 — FSC147 POSITIVE MANIFEST (NO GUESSING CLASS LABELS)
with open(ANNOTATION_FILE,encoding='utf-8') as f:
    ann=json.load(f)
with open(SPLIT_FILE,encoding='utf-8') as f:
    split_data=json.load(f)

class_map={}
with open(CLASS_FILE,encoding='utf-8-sig') as f:
    for line in f:
        line=line.strip()
        if not line:
            continue
        parts=line.split()
        if len(parts)>=2:
            class_map[Path(parts[0]).name]=' '.join(parts[1:]).strip()

images={p.name:str(p) for p in IMAGE_DIR.glob('*.jpg')}
rows=[]
missing=[]
for split in ['train','val','test']:
    for raw_name in split_data.get(split,[]):
        name=Path(str(raw_name)).name
        reasons=[]
        if name not in images: reasons.append('image')
        if name not in ann: reasons.append('annotation')
        if name not in class_map: reasons.append('class label')
        if reasons:
            missing.append((split,name,','.join(reasons)))
            continue
        points=ann[name].get('points',[])
        rows.append(dict(
            group_id=f'POS__{name}', pair_type='POSITIVE',
            image_name=name,image_path=images[name], target=class_map[name],
            source_class=class_map[name],gt_count=int(len(points)),
            presence_gt=1,split=split,origin_batch='FSC147_POSITIVE',
        ))

pos=pd.DataFrame(rows)
if pos.empty:
    raise RuntimeError('Không tạo được mẫu positive. Hãy kiểm tra ImageClasses và split.')
if pos.group_id.duplicated().any():
    raise RuntimeError('Một ảnh xuất hiện ở nhiều split; kiểm tra split file.')
if missing:
    print('MISSING',len(missing),'first 15',missing[:15])
    raise RuntimeError('Không được âm thầm bỏ ảnh FSC-147 thiếu dữ liệu!')

print('Positive pairs:',len(pos),'Images:',pos.image_name.nunique())
print('Classes:',pos.target.nunique(),'Splits:',pos.split.value_counts().to_dict())
print('GT range:',pos.gt_count.min(),pos.gt_count.max())
if len(pos)!=6135:
    print('CẢNH BÁO: số positive không bằng 6.135; chỉ tin theo số match thực tế.')
pos.to_csv(OUTPUT_DIR/'fsc147_full_positive_manifest.csv',index=False)


In [ ]:
# CELL 4 — 111 VERIFIED ABSENT PAIRS + THẨM ĐỊNH CHÉO
neg=pd.read_csv(ABSENT_CSV)
need={
    'pair_id','image_name','source_class','query_target','presence_label','count_target',
    'origin_batch','final_decision','benchmark_inclusion'
}
miss=sorted(need-set(neg.columns))
if miss: raise ValueError(f'111 CSV thiếu cột {miss}')

neg=neg[neg.benchmark_inclusion.astype(str).str.upper().eq('YES')].copy()
neg['presence_label']=pd.to_numeric(neg['presence_label'],errors='raise')
neg['count_target']=pd.to_numeric(neg['count_target'],errors='raise')
if len(neg)!=111 or not (neg.presence_label==0).all() or not (neg.count_target==0).all():
    raise RuntimeError('Bộ negative phải có đúng 111 cặp GT=0, presence=0.')
if not neg.final_decision.astype(str).eq('KEEP_NEGATIVE').all():
    raise RuntimeError('Có pair không phải KEEP_NEGATIVE.')
if neg.pair_id.duplicated().any():
    raise RuntimeError('Có pair_id lặp.')

pos_image_ref=pos.set_index('image_name')
absent_images=set(neg.image_name)
if not absent_images.issubset(set(pos.image_name)):
    raise RuntimeError('111 benchmark có ảnh không thuộc FSC-147 manifest.')

for _,r in neg.iterrows():
    p=pos_image_ref.loc[r.image_name]
    if str(p['target']).strip().casefold()!=str(r.source_class).strip().casefold():
        raise RuntimeError(f'Khác source_class với official image label: {r.image_name}: {p["target"]} vs {r.source_class}')
    if p['split']!='test':
        raise RuntimeError(f'Hard negative không thuộc official TEST: {r.image_name}, split={p["split"]}')

neg_manifest=pd.DataFrame({
    'group_id':neg.pair_id.astype(str),
    'pair_type':'HARD_NEGATIVE',
    'image_name':neg.image_name.astype(str),
    'image_path':neg.image_name.map(images),
    'target':neg.query_target.astype(str),
    'source_class':neg.source_class.astype(str),
    'gt_count':0,
    'presence_gt':0,
    'split':'test',
    'origin_batch':neg.origin_batch.astype(str),
})
neg_manifest.to_csv(OUTPUT_DIR/'verified_negative_111_manifest.csv',index=False)

joint=pd.concat([pos,neg_manifest],ignore_index=True)
if joint.group_id.duplicated().any(): raise RuntimeError('group_id lặp khi ghép manifest!')
if joint[['image_name','target']].duplicated().any():
    print('WARNING: Có image+target trùng, kiểm tra kỹ những prompt tương đương/ngữ nghĩa.')

joint['label_source']=np.where(joint.pair_type.eq('POSITIVE'),'FSC147_POINTS','MANUAL_AUDIT_ABSENT')
joint=joint.sort_values(['split','pair_type','image_name','target']).reset_index(drop=True)
joint.to_csv(OUTPUT_DIR/'v7_joint_full_manifest.csv',index=False)
joint[joint.split.eq('test')].to_csv(OUTPUT_DIR/'v7_joint_official_test_manifest.csv',index=False)

print('POSITIVE',len(pos),'NEGATIVE',len(neg_manifest),'TOTAL PAIRS',len(joint))
print('UNIQUE IMAGES',joint.image_name.nunique())
print('TEST positive',sum((joint.split=='test') & (joint.pair_type=='POSITIVE')))
print('TEST negative',sum((joint.split=='test') & (joint.pair_type=='HARD_NEGATIVE')))
print('111 source images',neg_manifest.image_name.nunique())
print('111 origin batches',neg_manifest.origin_batch.value_counts().to_dict())


In [ ]:
# CELL 5 — OPTIONAL: NẠP KẾT QUẢ V7 CŨ / MỚI
# CHỈ dùng v7_count thực sự do V7 suy luận, không lấy baseline_count.
# Nếu V7 được Add Input dưới dạng ZIP thì tự giải nén.
V7_ZIP=find_file('nckh_v7_dual_evidence_router_export.zip')
if V7_ZIP is not None:
    V7_CACHE_DIR=Path('/kaggle/working/v7_joint_cached_old_extract')
    V7_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(V7_ZIP,'r') as zf:
        zf.extractall(V7_CACHE_DIR)
    print('Extracted old V7 results from:',V7_ZIP)

def pred_files():
    found=[]
    for root in SEARCH_ROOTS:
        if not root.exists(): continue
        for p in root.rglob('*.csv'):
            if OUTPUT_DIR in p.parents: continue
            n=p.name.lower()
            if n=='v7_results.csv' or n=='v7_full_predictions.csv' or (n.startswith('v7_') and 'predictions' in n):
                found.append(p)
    return sorted(set(found))

prediction_parts=[]
for p in pred_files():
    try:
        x=pd.read_csv(p)
        if 'group_id' not in x.columns: continue
        col='v7_count' if 'v7_count' in x.columns else ('pred_count' if 'pred_count' in x.columns else None)
        if not col: continue
        piece=x[['group_id',col]].rename(columns={col:'v7_count'}).copy()
        piece['v7_count']=pd.to_numeric(piece['v7_count'],errors='coerce')
        piece=piece.dropna(subset=['group_id','v7_count'])
        piece['input_file']=str(p)
        prediction_parts.append(piece)
        print('Found V7 predictions',len(piece),'from',p)
    except Exception as e:
        print('Skipping bad CSV',p,repr(e))

if prediction_parts:
    pred_all=pd.concat(prediction_parts,ignore_index=True)
    # Allow identical duplicate predictions from repeated copies. Mismatch is an error.
    conflicts=pred_all.groupby('group_id').v7_count.nunique()
    if (conflicts>1).any():
        raise RuntimeError('Các output V7 khác nhau cho cùng group_id: '+str(conflicts[conflicts>1].index.tolist()[:10]))
    pred_all=pred_all.drop_duplicates('group_id',keep='last')
else:
    pred_all=pd.DataFrame(columns=['group_id','v7_count','input_file'])

if len(pred_all) and ((pred_all.v7_count<0).any() or (~np.isfinite(pred_all.v7_count)).any()):
    raise RuntimeError('Prediction phải là số hữu hạn không âm.')

joined=joint.merge(pred_all[['group_id','v7_count']],on='group_id',how='left',validate='one_to_one',indicator=True)
joined['has_prediction']=joined['v7_count'].notna()
joined.to_csv(OUTPUT_DIR/'v7_joint_predictions_status.csv',index=False)

pending=joined[~joined.has_prediction].drop(columns=['v7_count','_merge'])
pending.to_csv(OUTPUT_DIR/'v7_pending_predictions.csv',index=False)

if not 0<=SHARD_INDEX<NUM_SHARDS: raise ValueError('Bad SHARD_INDEX/NUM_SHARDS')
pending_shard=pending.iloc[SHARD_INDEX::NUM_SHARDS].copy()
pending_shard.to_csv(OUTPUT_DIR/f'v7_pending_shard_{SHARD_INDEX:03d}_of_{NUM_SHARDS:03d}.csv',index=False)

print('\n=== COVERAGE ===')
print('TOTAL',len(joined),'DONE',joined.has_prediction.sum(),'MISSING',len(pending))
print(joined.groupby(['pair_type','split']).has_prediction.agg(['sum','count']))
print('Pending shard size:',len(pending_shard))
if len(pending):
    print('IMPORTANT: V7 predictions are missing. Joint full evaluation is NOT complete.')


In [ ]:
# CELL 6 — EVALUATION: KHÔNG TRỘN NHẦM METRIC POSITIVE / NEGATIVE

def calc_metrics(g):
    if not len(g): return {}
    p=g.v7_count.to_numpy(float)
    y=g.gt_count.to_numpy(float)
    pos_mask=g.pair_type.eq('POSITIVE').to_numpy()
    neg_mask=~pos_mask
    result={
        'n':int(len(g)),
        'mae_micro_all_pairs':float(np.abs(p-y).mean()),
        'rmse_all_pairs':float(np.sqrt(np.mean((p-y)**2))),
        'presence_accuracy':float(np.mean((p>0)==(y>0))),
        'n_positive':int(pos_mask.sum()),
        'n_negative':int(neg_mask.sum()),
    }
    if pos_mask.any():
        pp=p[pos_mask]; yy=y[pos_mask]
        result['positive_mae']=float(np.mean(np.abs(pp-yy)))
        result['positive_fnr']=float(np.mean(pp==0))
        result['positive_survival']=float(np.mean(pp>0))
    if neg_mask.any():
        nn=p[neg_mask]
        result['hard_negative_mae']=float(np.mean(nn))
        result['hard_negative_fpr']=float(np.mean(nn>0))
        result['hard_negative_zero_rate']=float(np.mean(nn==0))
    if pos_mask.any() and neg_mask.any():
        result['macro_balanced_mae']=float((result['positive_mae']+result['hard_negative_mae'])/2)
    return result

rows=[]
scopes={
    'official_TEST_positive_plus_111_negative':joined[joined.split.eq('test')],
    'ALL_FSC147_positive_plus_111_negative':joined,
    'official_TEST_positive_only':joined[(joined.split=='test')&(joined.pair_type=='POSITIVE')],
    'ALL_positive_only':joined[joined.pair_type.eq('POSITIVE')],
    '111_negative_only':joined[joined.pair_type.eq('HARD_NEGATIVE')],
    'OLD_56_negative_only':joined[joined.origin_batch.eq('V3_OLD_56')],
    'NEW_55_negative_only':joined[joined.origin_batch.eq('V4_NEW_REVIEWED')],
}

for scope_name,g in scopes.items():
    n_expected=len(g)
    done=g[g.has_prediction]
    n_done=len(done)
    complete=n_done==n_expected
    row={
        'scope':scope_name,'n_expected':int(n_expected),'n_predicted':int(n_done),
        'coverage':float(n_done/n_expected) if n_expected else np.nan,
        'status':'COMPLETE' if complete else 'INCOMPLETE',
    }
    if n_done and (complete or not REQUIRE_COMPLETE_FOR_MAIN_RESULTS):
        row.update(calc_metrics(done))
    rows.append(row)

report=pd.DataFrame(rows)
report.to_csv(OUTPUT_DIR/'v7_joint_evaluation_summary.csv',index=False)
print('=== V7 EVALUATION SUMMARY ===')
display(report)

is_full_complete=bool(joined.has_prediction.all())
is_test_complete=bool(joined.loc[joined.split.eq('test'),'has_prediction'].all())
if not is_full_complete:
    print('FULL result chưa có vì còn',int((~joined.has_prediction).sum()),'cặp thiếu dự đoán V7.')
if not is_test_complete:
    print('Official TEST joint evaluation chưa thể công bố.')


In [ ]:
# CELL 7 — RESUME CHECK + VERIFY PREDICTION FILES (KHÔNG ĐẶT COUNT=0 KHI THIẾU)
# Đây là contract để chạy phần còn thiếu bằng V4→V5→V6→V7.
# Đầu ra tối thiểu của stage inference FULL phải có 2 cột:
#   group_id, v7_count

expected=set(joint.group_id)
received=set(pred_all.group_id)
not_in_benchmark=received-expected
print('Prediction IDs not in benchmark:',len(not_in_benchmark))
if not_in_benchmark:
    print('Examples:',sorted(not_in_benchmark)[:15])

# Lưu metadata để nhóm theo dõi nhãn và benchmark version.
metadata={
  'evaluation_name':'V7_FSC147_111_JOINT',
  'full_positive_pairs':int(len(pos)),
  'negative_pairs':int(len(neg_manifest)),
  'total_query_image_pairs':int(len(joint)),
  'unique_images':int(joint.image_name.nunique()),
  'official_test_positive_pairs':int(((joint.split=='test')&(joint.pair_type=='POSITIVE')).sum()),
  'negative_unique_images':int(neg_manifest.image_name.nunique()),
  'done':int(joined.has_prediction.sum()),
  'missing':int((~joined.has_prediction).sum()),
  'full_coverage':bool(is_full_complete),
  'official_test_coverage':bool(is_test_complete),
  'warning':'V7 cache from prior versions does not imply full FSC-147 inference. Missing predictions are not zero.',
  'research_note':'111 hard negatives were already used during V4-V7 development; these are NOT an untouched independent test.',
}
with open(OUTPUT_DIR/'v7_joint_metadata.json','w',encoding='utf-8') as f:
    json.dump(metadata,f,ensure_ascii=False,indent=2)
print(json.dumps(metadata,ensure_ascii=False,indent=2))


In [ ]:
# CELL 8 — EXPORT ZIP
import shutil
from IPython.display import FileLink, display

archive=shutil.make_archive(
    '/kaggle/working/nckh_v7_fsc147_111_joint_export',
    'zip',root_dir=str(OUTPUT_DIR)
)
print('EXPORTED',archive)
display(FileLink(archive))


## Hướng dẫn chạy V7 cho PENDING pairs

Để hoàn tất yêu cầu của nhóm trưởng, **phải chạy đúng các stage trước khi áp dụng V7**:

1. V4: CountGD++ candidate detection + Qwen3-VL box verification cho mỗi `group_id` chưa có kết quả.
2. V5: confounder discovery và chạy lại CountGD++ với negative text/exemplar.
3. V6: Qwen target-evidence router + count cache.
4. V7: dual-evidence router lấy `baseline_count`, `v4_filtered_count`, `neg_text_count`, `neg_text_exemplar_count`, `target_evidence`, `keep_boxes_shown`, `decision` để tính `v7_count`.
5. Lưu từng shard thành file `v7_full_predictions.csv` hoặc `v7_shard_predictions_XX.csv` gồm `group_id,v7_count`.
6. Add lại các CSV vào Kaggle Input, chạy lại notebook này. Khi **coverage = 100%**, bảng thống kê mới là kết quả của full evaluation.

**Lưu ý:** V7 trước đó là router dựa trên cache V4–V6, không phải mô hình độc lập có thể gọi trực tiếp qua `model.predict(image, text)`. Nếu chỉ có `v7_results.csv` từ 207 pairs cũ, notebook sẽ báo những positive mới còn thiếu, chứ không tạo số liệu giả.
